Hello! this is a tutorial on how to make and modify orbits using JKAT.

First step is importing jkat, alongside some other useful packages:

In [ ]:
import jkat
import numpy as np
import matplotlib.pyplot as plt
import datetime as dt

Next step is to create an orbit. JKAT stores orbits as 2D Keplerian orbits (`jkat.Orbit`) defined by 6 orbital elements.
The elements in question are:
- the Parameter $p$, `Orbit.p`
- the Eccentricity $e$, `Orbit.e`
- the Inclination $i$, `Orbit.i`
- the Right Ascention of the Ascending Node $\Omega$, `Orbit.raan` 
  (also called Longitude of the Ascending Node) 
- the Argument of Periapsis $\omega$, `Orbit.argp`
- the Time of Periapsis Passage $t_p$, `Orbit.tp`

Finally, to represent what body the orbit is around, a Standard Gravitational Parameter $\mu$ needs to be supplied.
For small object such as satellites, this is simply the gravitational constant times the mass of the body $\mu = GM$.

(if you are wondering why we use the parameter $p$ instead of the semi-major-axis $a$, that is because $p$ behaves more nicely with orbits where the eccentricity is close to $e=1$, we'll see later how the semi major axis can be used instead if you prefer)

Now, let's make out first orbit:

As our example, let's use the Galileo-IOV FM1, the first satellite in the ESA Gallileo constellation.
Not accounting for perturbations (beyond the scope of this notebook), it's orbital elements are:
- parameter = 29508 km
- eccentricity = 0.0002931
- inclination = 56.9925 degrees
- right ascention of ascending node = 341.627 degrees
- argument of perigee = 54.4577 degrees
- time of periapsis passage 2026-06-22 15:10:13

to create an orbit we simply enter this into the orbit class:

In [ ]:
parameter = 29508 # [km]
eccentricity = 0.0002931 # [-]
inclination = np.radians(56.9925) # [deg]
right_ascention_ascending_node = np.radians(341.627) # [deg]
argument_perigee = np.radians(54.4577) # [deg]
time_periapsis = dt.datetime(2026,6,22, 15,10,13)
time_periapsis = jkat.to_time(time_periapsis) # we convert the date into seconds

ob = jkat.Orbit(
        p = parameter,
        e = eccentricity,
        i = inclination,
        raan = right_ascention_ascending_node,
        argp = argument_perigee,
        tp = time_periapsis,
        mu = jkat.EARTH_MU
)

print(ob)

As you can see, the orbit has been nicely defined. now what can we do with it?

Firstly, we can access some resulting parameters of the orbit:

In [ ]:
print(f"Period in hours: {ob.T / 3600}")
print(f"period in days: {ob.period / 86_000}") # parameters can be accessed by long and short names
print(f"semi-major axis: {ob.a} km")
print(f"True anomaly right now: {ob.f(jkat.to_time(dt.datetime.now()))} radians")


In general, we can access a range of parameters from the orbit. some parameters also have a long form alias. Some examples of accessible parameters is:
- semi major axis, `a` or `semi_major_axis`
- periapsis and apoapsis, `pe`,`ap` or `periapsis`,`apoapsis`
- angular momentum, `h` or `angular_momentum`
- longitude of periapsis `longp` or `longitude_periapsis`
- orbital period, `T` or `period`

Some variables change depending on where in the orbit the orbiting body is.
these can be accessed by providing the true anomaly $f$ or the time $t$ you want to know the variable at.
for example, the velocity and altitude:

In [ ]:
ff = np.linspace(0,2*np.pi,100) # range of true anomalies
rr = ob.r(ff) # radius from center of orbit
rr -= jkat.EARTH_RADIUS # subtract earth radius to get altitude
plt.xlabel('true anomaly'); plt.ylabel("altitude [km]")
plt.plot(ff,rr)
plt.show()

vv = ob.v(ff)
plt.xlabel('true anomaly'); plt.ylabel("velocity [km/s]")
plt.plot(ff,vv)
plt.show()

t_apoapsis = ob.t(np.pi)
print(f"time at apoapsis: {t_apoapsis}")
f_at_epoch = ob.f(0) # time 0 is 1st of january 2000 CE
print(f"true anomaly at time 0: {f_at_epoch}")




Parameters can also be changed. Geosynchronous altitude is about 35,786 km above the earths surface, and any orbit at that altitude will take exactly one (sidereal) day to complete. We can change the semi-major axis of our orbit to match this number

In [ ]:
ob.a = jkat.EARTH_RADIUS + 35_786
ob.T / jkat.SIDEREAL_DAY

Oops, looks like that number was not entirely correct (the bulge of the equator adds a few km to the Earths radius), so let's change the period itself instead

In [ ]:
ob.T = jkat.SIDEREAL_DAY
ob.a - jkat.EARTH_RADIUS

That's better, in general, all parameters can be modified if there is a clear interpretation of how it would affect the other parameters.
for instance changing the apoapsis via `ap` will change the eccentricity and parameter to leave the periapsis the same.

one final type of parameter that can be accesed is vectorized versions of the distance and velocity.
These can be accessed by true anomaly as `ob.rvec(f)`, `ob.vvec(f)`, or by time `ob.t2rvec(t)`, `ob.t2vvec(t)`.
if both are needed then they can be accessed via `ob.vectors(f)` or `ob.t2vectors(t)`.
Let's use this to find the how distance to the Moon from our satellite changes over a year:

In [ ]:
Moon = jkat.Moon # several example orbits are provided
tt = np.linspace(
    jkat.to_time(dt.date(2026,1,1)),
    jkat.to_time(dt.date(2026,2,1)),
    400
)
distances = []
for t in tt:
    r_moon = Moon.t2rvec(t)
    r_ob = ob.t2rvec(t)
    r_dist = r_moon - r_ob
    distances.append(np.linalg.norm(r_dist))

plt.plot((tt- tt[0])/86_000, distances)
plt.xlabel("day")
plt.ylabel("distance [km]")
plt.show()

And that's the basics of making and modifying orbits. There are several other ways of making and modifying orbits. But this should be enough for a start